# Configuração dos parâmetros

In [0]:
catalog = "olist_medalion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Funções auxiliares

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Flag de negócio
APENAS_LANCADOS = True

# Função auxiliar para geração de Surrogate Key (SK) determinística
def sk_hash(*cols):
    return (F.conv(F.substring(F.sha2(F.concat_ws("||", *cols), 256), 1, 15), 16, 10)
            .cast("bigint"))

## Dim_movies

In [0]:
df_info = spark.table(f"{silver_schema}.tb_info_filmes").dropDuplicates(["id_filme"])

# Transformação
df_dim_movies = (
    df_info
    .select(
        sk_hash(F.col("id_filme")).alias("sk_movie_id"),
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse")
    )
)

# Escrita Gold
df_dim_movies.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} registros")

## Dim_Genres

In [0]:
df_gen = spark.table(f"{silver_schema}.tb_generos")

# Transformação
df_dim_genres = (
    df_gen
    .select(F.col("genero").alias("nome_genero"))
    .distinct()
    .select(
        sk_hash(F.col("nome_genero")).alias("sk_genre_id"), 
        F.col("nome_genero")
    )
)

# Escrita Gold
df_dim_genres.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"dim_genres: {df_dim_genres.count()} registros")

## Dim_people

In [0]:
df_ent = spark.table(f"{silver_schema}.tb_pessoas_empresas")

# Transformação
df_dim_people = (
    df_ent
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    .select(
        sk_hash(F.col("nome_pessoa"), F.col("tipo_pessoa")).alias("sk_person_id"),
        F.col("nome_pessoa"), 
        F.col("tipo_pessoa")
    )
)

# Escrita Gold
df_dim_people.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"dim_people: {df_dim_people.count()} registros")

## Dim_Companies

In [0]:
df_ent = spark.table(f"{silver_schema}.tb_pessoas_empresas")

# Transformação
df_dim_companies = (
    df_ent
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .select(
        sk_hash(F.col("nome_produtora")).alias("sk_company_id"), 
        F.col("nome_produtora")
    )
)

# Escrita Gold
df_dim_companies.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"dim_companies: {df_dim_companies.count()} registros")

## Dim_reviews

In [0]:
df_rev = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

# Transformação
df_dim_reviews = (
    df_rev
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg(F.col("nota_usuario").cast("double")), 2).alias("nota_media_usuarios")
    )
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .select(
        sk_hash(F.lit("review"), F.col("id_filme")).alias("sk_review_id"),
        F.col("sk_movie_id"), 
        F.col("qtd_avaliacoes_usuarios"), 
        F.col("nota_media_usuarios")
    )
)

# Escrita Gold
df_dim_reviews.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"dim_reviews: {df_dim_reviews.count()} registros")

# Fact_movies_performance

In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes").dropDuplicates(["id_filme"])
df_met = spark.table(f"{silver_schema}.tb_metricas_engajamento").dropDuplicates(["id_filme"])

# Filtro de negócio
df_base_fato = df_dim_movies.select("sk_movie_id", "id_filme", "status_filme")
if APENAS_LANCADOS:
    df_base_fato = df_base_fato.filter(F.col("status_filme") == "Lançado")

# Transformação
df_fact_movies_performance = (
    df_base_fato
    .join(df_fin, "id_filme", "left")
    .join(df_met, "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.expr("try_cast(qtd_votos_tmdb as int)").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.expr("try_cast(qtd_votos_imdb as int)").alias("qtd_votos_imdb")
    )
)

# Escrita Gold
df_fact_movies_performance.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact_movies_performance: {df_fact_movies_performance.count()} registros")

## Bridge_movies_genre

In [0]:
df_gen = spark.table(f"{silver_schema}.tb_generos")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")

# Transformação
df_bridge_movie_genre = (
    df_gen
    .join(df_dim_movies, "id_filme", "inner")
    .join(df_dim_genres, df_gen["genero"] == df_dim_genres["nome_genero"], "inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

# Escrita Gold
df_bridge_movie_genre.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

print(f"bridge_movie_genre: {df_bridge_movie_genre.count()} registros")

## Bridge_movie_person

In [0]:
df_ent = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

# Transformação
df_bridge_movie_person = (
    df_ent
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(df_dim_movies, "id_filme", "inner")
    .join(
        df_dim_people,
        (F.col("nome_entidade") == df_dim_people["nome_pessoa"]) &
        (F.col("tipo_entidade") == df_dim_people["tipo_pessoa"]),
        "inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

# Escrita Gold
df_bridge_movie_person.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

print(f"bridge_movie_person: {df_bridge_movie_person.count()} registros")

## Bridge_movie_company

In [0]:
df_ent = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

# Transformação
df_bridge_movie_company = (
    df_ent
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(df_dim_movies, "id_filme", "inner")
    .join(df_dim_companies, F.col("nome_entidade") == df_dim_companies["nome_produtora"], "inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

# Escrita Gold
df_bridge_movie_company.write.format("delta").mode("append") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print(f"bridge_movie_company: {df_bridge_movie_company.count()} registros")

# Perguntas de Negócio

In [0]:
fact       = spark.table(f"{gold_schema}.fact_movies_performance")
dim_movies = spark.table(f"{gold_schema}.dim_movies")
dim_genres = spark.table(f"{gold_schema}.dim_genres")
dim_people = spark.table(f"{gold_schema}.dim_people")
dim_comp   = spark.table(f"{gold_schema}.dim_companies")
br_genre   = spark.table(f"{gold_schema}.bridge_movie_genre")
br_person  = spark.table(f"{gold_schema}.bridge_movie_person")
br_comp    = spark.table(f"{gold_schema}.bridge_movie_company")

# Receita total somada de todos os filmes da base
df_q1 = fact.agg(F.sum("receita_brl").alias("receita_total_brl"))
display(df_q1)

# Top 5 filmes com maior popularidade
df_q2 = (
    fact.join(dim_movies, "sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(F.desc("popularidade"), F.asc("titulo"))
    .limit(5)
)
display(df_q2)

# Quantidade de filmes por gênero (maior para menor)
df_q3 = (
    br_genre.join(dim_genres, "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"), F.asc("nome_genero"))
)
display(df_q3)

# Top 10 filmes de maior receita, com RANK()
df_q4 = (
    fact.join(dim_movies, "sk_movie_id")
    .filter(F.col("receita_usd").isNotNull())
    .withColumn("ranking", F.rank().over(Window.orderBy(F.desc("receita_usd"))))
    .filter(F.col("ranking") <= 10)
    .select("ranking", "titulo", "receita_usd", "receita_brl")
    .orderBy("ranking", "titulo")
)
display(df_q4)

# Data limite dos recortes: lançamento mais recente entre filmes lançados,
# ignorando datas futuras e nulas
df_lancados = dim_movies.filter(
    (F.col("status_filme") == "Lançado")
    & F.col("data_lancamento").isNotNull()
    & (F.col("data_lancamento") <= F.current_date())
)

data_limite = df_lancados.agg(F.max("data_lancamento")).collect()[0][0]
print(f"Data limite (lançamento mais recente): {data_limite}")


def filmes_ultimos_anos(anos):
    return df_lancados.filter(
        F.col("data_lancamento").between(F.add_months(F.lit(data_limite), -12 * anos),
                                         F.lit(data_limite))
    ).select("sk_movie_id")


# Ator com mais participações nos filmes dos últimos 2 anos
df_atores = (
    br_person
    .join(filmes_ultimos_anos(2), "sk_movie_id")
    .join(dim_people.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
)

df_q5 = (
    df_atores
    .withColumn("ranking", F.rank().over(Window.orderBy(F.desc("qtd_participacoes"))))
    .filter(F.col("ranking") == 1)
    .select("nome_pessoa", "qtd_participacoes")
    .orderBy("nome_pessoa")
)
display(df_q5)

# Produtora com maior lucro nos últimos 5 anos
df_prod = (
    br_comp
    .join(filmes_ultimos_anos(5), "sk_movie_id")
    .join(fact, "sk_movie_id")
    .join(dim_comp, "sk_company_id")
    .groupBy("sk_company_id", "nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"),
         F.sum("lucro_brl").alias("lucro_total_brl"),
         F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .filter(F.col("lucro_total_usd").isNotNull())
)

df_q6 = (
    df_prod
    .withColumn("ranking", F.rank().over(Window.orderBy(F.desc("lucro_total_usd"))))
    .filter(F.col("ranking") == 1)
    .select("nome_produtora", "lucro_total_usd", "lucro_total_brl", "qtd_filmes")
)
display(df_q6)